## Calibrate Hidden Cost Sensitivity

**Purpose**
Runs hidden-cost calibration scenarios and aggregates hidden cost outputs across scenario variants.

**Primary inputs**
- `project/config/hidden_cost/*.json`
- `model outputs from runs/hidden_cost_sensitivity/*`

**Primary outputs**
- `data/hidden_cost_result.csv`
- `plots shown in notebook`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# Hidden Cost Sensitivity

**Purpose:** Run sensitivity scenarios for hidden costs (discount rates, financing impacts on retrofit incentives).

**Project imports:** `project.model`.

**Inputs:** Reference config, hidden cost parameters.

**Outputs:** 7 scenario runs under `runs/hidden_cost_sensitivity/` and aggregated table in `data/hidden_cost_result.csv`.


In [ ]:
import os
import pandas as pd
from project.coupling import ini_res_irf


In [ ]:
folder = os.path.join('project', 'config', 'hidden_cost')

# 'discount_high': {'config': 'config_discount_rate_high.json', 'path': 'runs/hidden_cost_sensitivity/discount_high'},
# 'discount_upper': {'config': 'config_discount_rate_upper.json', 'path': 'runs/hidden_cost_sensitivity/discount_upper'},

configs = {
    'discount_upper': {'config': 'config_discount_rate_upper.json', 'path': 'runs/hidden_cost_sensitivity/discount_upper'},
    'discount_constant': {'config': 'config_discount_rate_medium.json', 'path': 'runs/hidden_cost_sensitivity/discount_constant'},
    'discount_high': {'config': 'config_discount_rate_high.json', 'path': 'runs/hidden_cost_sensitivity/discount_high'},
    'reference': {'config': 'config.json', 'path': 'runs/hidden_cost_sensitivity/reference'},
    'no_financing': {'config': 'config_no_financing.json', 'path': 'runs/hidden_cost_sensitivity/no_financing'},
    'frequency_5': {'config': 'config_frequency_5.json', 'path': 'runs/hidden_cost_sensitivity/frequency_5'},
    'frequency_10': {'config': 'config_frequency_10.json', 'path': 'runs/hidden_cost_sensitivity/frequency_10'},
}

scenarios = ['reference']
configs_run = {key: item for key, item in configs.items() if key in scenarios}




In [ ]:
for key, item in configs_run.items():
    print(key)
    ini_res_irf(config=os.path.join(folder, item['config']), path=item['path'])


In [ ]:
result = {}
for key, item in configs.items():
    hidden_cost = pd.read_csv(item['path'] + '/calibration/hidden_cost_insulation.csv', index_col=[0, 1, 2, 3])
    result.update({key: hidden_cost.copy()})
    
result = pd.concat(result, axis=1)   
result.columns.names = ['Scenario', 'Type']
os.makedirs('data', exist_ok=True)
result.to_csv(os.path.join('data', 'hidden_cost_result.csv'))


In [ ]:
temp = result.xs('Hidden cost', level='Type', axis=1)

names = list(temp.index.names)
def combine_levels(row):
    return ", ".join([name for name, value in zip(names, row.name) if value])

# Transforming the index levels into a single level
temp.index = temp.apply(combine_levels, axis=1)
temp.index.names = ['Measures']
temp = temp.stack().squeeze()
temp.name = 'Value'
temp = temp.reset_index()
temp = temp.sort_values(by='Value', ascending=False)
temp['Value'] = -temp['Value'] / 1e3


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

sns.stripplot(data=temp, x="Measures", y="Value", hue='Scenario', jitter=False, ax=ax)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_visible(True)

ax.set_ylabel('Hidden cost [k€]')

ax.legend(loc='upper left', bbox_to_anchor=(1, 1), frameon=False)

plt.xticks(rotation=90)
plt.show()

